In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import fnmatch
from connect import bob
from utils import *
from limb_fitting import *
from scipy.ndimage import gaussian_filter

In [2]:
def reflection_point_predict(header):
    px = [1.63114715e-06, 6.72511045e-03, 9.60448053e+02]
    py = [4.61830880e-06, -6.85005911e-03, 9.77508840e+02]

    r_sun = header['RSUN_ARC']
    dx, dy = header['PXBEG2'] - 1, header['PXBEG1'] - 1

    xr = np.polyval(px, r_sun) - dx
    yr = np.polyval(py, r_sun) - dy
    return xr, yr


def roll(image, dx, dy):
    nx, ny = image.shape
    image_ = np.zeros_like(image)
    x, y = int(round(dx)), int(round(dy))
    image_[max(x, 0): min(nx + x, nx), max(y, 0): min(ny + y, ny)] = image[max(-x, 0): min(nx - x, nx),
                                                                     max(-y, 0): min(ny - y, ny)]
    return image_

def reflect(image, xr, yr):
    nx, ny = image.shape
    return roll(image[::-1, ::-1], 2 * int(round(xr)) - nx + 1, 2 * int(round(yr)) - ny + 1)

In [58]:
sftp = bob()

top_dir = '/data/slam/valori/test_l2_fmdb/FDT_test_release_v08_2025/v4/l2/'
#top_dir = '/data/solo/phi/data/fmdb/l1/'
dirs = sorted(sftp.listdir(top_dir))

Q = []

for directory in dirs:
    if fnmatch.fnmatch(directory, '2024-01*'):
        for file in sorted(sftp.listdir(top_dir + directory)):
            #if fnmatch.fnmatch(file, '*fdt-[ia]lam*.fits.gz'):
            if fnmatch.fnmatch(file, '*stokes*.fits.gz'):
                try:
                    print(file)

                    remote_file = top_dir + directory + '/' + file
                    local_file = 'temp.fits.gz'
                    sftp.get(remote_file, local_file)
                except:
                    pass

                stop

solo_L2_phi-fdt-stokes_20240101T040003_V202602220902_0441010503.fits.gz


NameError: name 'stop' is not defined

In [3]:
s = np.load('/home/ulyanov/data/solo/phi/distortion/fdt/distortion_cor.npz')
xd, yd = s['xd'], s['yd']

with fits.open('temp.fits.gz') as hdul:
    header = hdul[0].header
    data = hdul[0].data

cpos = header['CONTPOS'] - 1
xr, yr = reflection_point_predict(header)

image = data[0,0].copy()
ghost = gaussian_filter(reflect(image, xr, yr), 8)
image = image - ghost * 0.01

image = undistort(image, header, xd, yd)

In [4]:
plt.figure(figsize=(10,10))
plt.imshow(image, vmin=-0.01, vmax=0.01)

In [86]:
image_ = remove_straylight(image, beta=1.4)

In [87]:
plt.figure(figsize=(10,10))
plt.imshow(image_, vmin=-0.01, vmax=0.01)

In [84]:
image_ = deconvolve(image, epsilon=0.2, alpha=2., beta=1.4, radius=0.1, sigma=0.1, niter=10)

In [85]:
plt.figure(figsize=(10,10))
plt.imshow(image_, vmin=-0.01, vmax=0.01)

In [6]:
from fit_cld import *

r, q, params = fit_cld(image)
sigma, alpha, beta, epsilon, scale, bias, rsun = params
print(params)

[8.94283963e-01 2.27937761e+00 9.77664357e-01 2.56904048e-01
 9.91879512e-01 3.03983962e-03 4.55199997e+02]


In [7]:
plt.figure(figsize=(10,10))
plt.plot(r, q)
#plt.plot(r, neckel(np.sqrt((1 - (r / rsun) ** 2).clip(0))))
plt.plot(r, model(r, *params))

plt.xlim(rsun-20,rsun+20)
plt.ylim(0,0.6)
plt.grid(True)
plt.tight_layout()

In [38]:
mu = 12 #0.5 / 3600 * np.pi / 180 * 5e6
0.6173 * 35 / mu

1.8004583333333333

In [41]:
0.6173 * 33 / 10

2.03709

In [25]:
from astropy.modeling.functional_models import AiryDisk2D
nx, ny = data.shape[-2:]
xi, yi = np.mgrid[-10:10,-10:10]

airy = AiryDisk2D(radius=2.5 * 0.9)
q = airy(xi, yi)

In [26]:
plt.figure(figsize=(10,10))
plt.imshow(q)

In [27]:
q[10,10] ** 2 / np.sum(q ** 2), q[10,10] / np.sum(q)

(np.float64(0.5020212414906072), np.float64(0.23883375845739088))